In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

GM = 1.0


def kepler(t, state):
    x, y, vx, vy = state
    r = np.sqrt(x**2 + y**2)
    ax = -GM * x / r**3
    ay = -GM * y / r**3
    return [vx, vy, ax, ay]


# initial condition intended to give a bound ellipse
x0, y0 = 1.0, 0.0
vx0, vy0 = 0.0, 0.8
state0 = [x0, y0, vx0, vy0]

t_span = (0, 20)

sol = solve_ivp(kepler, t_span, state0,
                t_eval=np.linspace(0, 20, 1000),
                rtol=1e-9, atol=1e-9)

t = sol.t
x, y, vx, vy = sol.y
r = np.sqrt(x**2 + y**2)

# specific orbital energy (per unit mass)
E = 0.5 * (vx**2 + vy**2) - GM / r
E0 = E[0]

print(f"E(0)          = {E0:.12f}   ({'bound' if E0 < 0 else 'unbound'})")
print(f"max |E - E0|  = {np.max(np.abs(E - E0)):.3e}")
print(f"relative drift= {np.max(np.abs((E - E0) / E0)):.3e}")

fig, axes = plt.subplots(1, 2, figsize=(15, 9))

# 1) orbit
axes[0].plot(x, y)
axes[0].plot(0, 0, 'ko', ms=4, label='focus (mass)')
axes[0].axis('equal')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
axes[0].set_title('Kepler orbit')
axes[0].legend()


# 2) energy error, where the integrator's behaviour is actually visible
axes[1].plot(t, E - E0)
axes[1].set_xlabel('t'); axes[1].set_ylabel(r'$E(t) - E(0)$')
axes[1].set_title('Energy drift (conservation check)')
axes[1].ticklabel_format(axis='y', style='sci', scilimits=(0, 0))

plt.tight_layout()
plt.show()